In [18]:
from pathlib import Path

import dask.dataframe as dd


class XeniumBundle:
    def __init__(self, path: Path):
        self.path = Path(path)

    def validate(self) -> None:
        if not self.path.exists():
            raise FileNotFoundError(f"Xenium output bundle not found: {self.path}")
        if not self.path.is_dir():
            raise ValueError(f"Expected a Xenium output bundle directory: {self.path}")
        transcript_file = self.path / "transcripts.parquet"
        if not transcript_file.exists():
            raise FileNotFoundError(f"transcripts.parquet not found in Xenium bundle: {self.path}")

    @property
    def transcript_file(self) -> Path:
        return self.path / "transcripts.parquet"

    def load_dataframe(self) -> dd.DataFrame:
        self.validate()
        try:
            return dd.read_parquet(self.transcript_file)
        except Exception as e:
            raise ValueError(f"Unable to read {self.transcript_file} as a parquet file.") from e

In [19]:
bundle = XeniumBundle("/Users/priyaltripathi/SubQCAT/data/Xenium_Prime_Mouse_Brain_Coronal_FF_outs")

In [20]:
df = bundle.load_dataframe()

In [21]:
df_quality = df[df['qv']>= 20.0]

In [22]:
df_gene = df_quality[df_quality['is_gene']==True]

In [23]:
df_clean = df_gene[df_gene['cell_id'] != "UNASSIGNED"]

In [24]:
df_clean = df_gene[df_gene['cell_id'] != "-1"]

In [25]:
df = df_clean.drop(columns=["codeword_index", "codeword_category", "is_gene"])

In [26]:
df.head()

,transcript_id,cell_id,overlaps_nucleus,feature_name,x_location,y_location,z_location,qv,fov_name,nucleus_distance
0,281509337012024,UNASSIGNED,0,Abraxas1,210.187500,248.703125,16.781250,40.0,A6,221.125000
1,281509336597263,UNASSIGNED,0,Acox1,139.203125,242.156250,16.781250,40.0,A6,282.109375
2,281509336942433,UNASSIGNED,0,Aldoa,72.250000,30.734375,16.890625,40.0,A6,473.859375
3,281509336546623,UNASSIGNED,0,Alg13,240.703125,4.640625,16.671875,40.0,A6,411.781250
4,281509336788188,UNASSIGNED,0,Brd2,212.531250,80.500000,16.703125,40.0,A6,353.453125


In [29]:
unique_cells = df['cell_id'].unique().compute()


In [30]:
unique_cells

0       kigodgcl-1
1       kihakoko-1
2       kiheajii-1
3       adbmhdnb-1
4       adaibcno-1
           ...    
3784    odagldgo-1
3785    gdfhmjek-1
3786    gdihnmmf-1
3787    gdiplgmg-1
3788    gdjalldg-1
Name: cell_id, Length: 63163, dtype: string

In [27]:
print(df.columns)

Index(['transcript_id', 'cell_id', 'overlaps_nucleus', 'feature_name',
       'x_location', 'y_location', 'z_location', 'qv', 'fov_name',
       'nucleus_distance'],
      dtype='str')


In [28]:
len(df.index)

149297699